# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)
# Access metadata (as an object)
md_json = dataset.metadata.to_json()
# Print general dataset information
print(f"{md_json['name']}: {md_json['description']}")

## 2. Data Overview
Review available record sets, fields, their names, and `@id`s.


In [ ]:
# List all record sets, their @id, and their fields
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}")

for rs in record_sets:
    print(f"\nRecordSet: {rs.name} (@id={rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id={field.id}) [type={field.data_type}]")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. (We use the record set and field `@id`s identified above.)

In [ ]:
# For demonstration, list all record_set @id values
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("All available record_set @id:", record_set_ids)

# Load all record sets into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record_set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns in {record_set_id}: ", list(df.columns))
        print(df.head(2))
    else:
        print("  No records found.")

## 4. Exploratory Data Analysis (EDA)
We'll apply basic EDA to a suitable record set. Select a numeric field for filtering, normalization, and grouping. Make sure to use fields' `@id` as columns.

We'll select the main protein record set if available (commonly containing abundance/quantification columns). Update the variables below with actual `@id` as seen above.

In [ ]:
# Please update these variables for your use case as identified above.
import numpy as np

# Pick the first record set (likely main table)
if dataframes:
    primary_record_set_id = list(dataframes.keys())[0]
    df = dataframes[primary_record_set_id]
    print(f"Using primary record set: {primary_record_set_id}")
    print("Available columns (field @id):", list(df.columns))
    # Try to identify a numeric field for demonstration (modify as needed)
    potential_numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(potential_numeric_cols) > 0:
        numeric_field_id = potential_numeric_cols[0]
    else:
        # Attempt to coerce possible numeric columns
        for col in df.columns:
            try:
                _ = pd.to_numeric(df[col])
                numeric_field_id = col
                df[col] = pd.to_numeric(df[col], errors='coerce')
                break
            except Exception:
                continue
    print(f"Selected numeric field: {numeric_field_id}")
    # Filtering on numeric field
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (showing top 5):")
    print(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nFirst 5 normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try grouping by first non-numeric field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field_id = col
            break
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No loaded dataframes to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the chosen numeric field's distribution, and (if available) mean value by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals():
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id} (filtered)')
    plt.xlabel(numeric_field_id)
    plt.show()
    # If group info exists, plot group means
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=60, ha='right')
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR-superscript-2 dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the `mlcroissant` library. We accessed metadata, performed overview and inspection of record sets and fields by their `@id`, filtered and normalized data, applied basic grouping, and visualized distributions. This provides a foundation for further biological or data science analysis of the protein data.